# XLeRobot Digital Twin — Data, Streaming, Aggregation

**Course:** RBB2013 Digital Twin (May 2026)
**Team:** Aiman (lead), Bento, Ariq, Ibrohim, Raziq
**Rubric:** Project Data, Streaming, Aggregation (5%)

> *"Define digital twin state relevant to your digital twin problem. Demonstrate streaming acquisition of multiple real data streams and its aggregation, conversion to digital twin state."*

## 1. Digital twin state — formal definition

At any instant `t`, the digital twin's state `S(t)` is the tuple:

$$S(t) = \big(\; \text{joints}[6],\;\; \text{ee\_pose}(x, y, z),\;\; \text{target}(x, y, z, r),\;\; \text{obstacles}[\ldots],\;\; ts \;\big)$$

| State component | Meaning | Source | Storage |
|-----------------|---------|--------|---------|
| `joints[6]` | 6 joint angles (rad) | Omniverse sim tick | Redis `state:latest` + Timescale `robot_state.joints` |
| `ee_pose(x,y,z)` | End-effector position (cm) | Forward kinematics on joints | Same |
| `target(x,y,z,r)` | Scene target ball | extension.py `TARGET_BALL` | Same |
| `obstacles[…]` | Named obstacle spheres | extension.py `OBSTACLES` | Same |
| `ts` | Wall clock timestamp | Publisher | Timescale partition key |

**Two storages, two purposes:**
- **Redis for reads that only care about *now*** — dashboards, oncall, "where is the arm right now?". One key (`state:latest`), O(1) reads, overwritten every ~100 ms.
- **TimescaleDB for reads that care about *history*** — Grafana time-series, regression comparisons, incident forensics. Hypertable partitioned by `ts`, cheap time-range queries at any scale.

Pydantic v2 contract in `services/shared/schemas.py::SimState` — frozen v1.0 in sprint 1.

In [ ]:
# The digital twin state, as a pydantic contract
from services.shared.schemas import SimState, TargetPose
print(SimState.model_json_schema())

## 2. Multiple real data streams

Four concurrent streams flow into the digital twin. All are **real** — no mocked inputs in the live system.

| # | Stream | Producer | Transport | Rate | Consumer |
|---|--------|----------|-----------|------|----------|
| 1 | **User commands** | External clients (curl / demo UI) | HTTP `POST /command` | on-demand | `nl-command` |
| 2 | **LLM completions** | Ollama | HTTP JSON | per-command | `nl-command` (parses into `TargetPose`) |
| 3 | **Joint frames** | `dispatcher` (interpolator) | ZMQ PUSH/PULL | 30 fps × ~1 s per command | `sim-bridge` (Omniverse) |
| 4 | **Sim state** | `sim-bridge` extension | ZMQ PUB/SUB | 10 Hz continuous | `telemetry` |

Streams 3 and 4 form the tight simulation loop. Stream 1 triggers new motion. Stream 2 is the AI intent decode.

Tests use containerized real infrastructure (mosquitto, Timescale, Redis) rather than stubs — see `tests/integration/`.

## 3. Streaming acquisition mechanics

### 3.1 Command → LLM → planner → dispatcher (HTTP chain, event-driven)

```
POST /command             (external)
  │
  ▼
nl-command  ──►  Ollama :11434     (async httpx)
  │
  ▼
motion-planner /plan               (sync HTTP)
  │
  ▼
dispatcher /dispatch               (sync HTTP)
```

Each hop is an HTTP+JSON request. `services/shared/schemas.py` gives all four services the same pydantic v2 types, so contract violations fail fast with HTTP 422.

### 3.2 Dispatcher → sim-bridge (ZMQ PUSH/PULL, high-frequency)

- Sim-bridge (Omniverse extension) **binds** `PULL` on `tcp://*:5556` — it owns the endpoint.
- Dispatcher **connects** `PUSH` to `tcp://host.docker.internal:5556`.
- For every dispatched target, dispatcher sends **30 frames of interpolated joint angles** as JSON messages `{"joints": [...], "frame_id": i}`.
- Sim-bridge drains the PULL queue every Kit update tick, applies the newest received joint angles, discards older ones (prevents lag if the sim is slower than dispatch).

### 3.3 Sim-bridge → telemetry (ZMQ PUB/SUB, continuous streaming)

- Sim-bridge **binds** `PUB` on `tcp://*:5557`.
- Telemetry **connects** SUB to `tcp://host.docker.internal:5557`.
- At 10 Hz, sim-bridge publishes the full `SimState` snapshot — joints + target + obstacles + implicit ts.
- Publishing is non-blocking (`zmq.NOBLOCK`) so a slow subscriber does not stall the sim tick.

## 4. Aggregation — from raw stream to digital twin state

Two aggregations happen inside `services/telemetry/app.py`.

### 4.1 Historical aggregation → TimescaleDB

```python
insert_state(pg_conn, state)   # INSERT INTO robot_state (ts, joints, ee_x, ee_y, ee_z) VALUES (...)
```

- One SQL row per received ZMQ message.
- Table is a **TimescaleDB hypertable** (partitioned by `ts`) — cheap time-range queries, automatic partitioning, gigabytes of history without index bloat.
- Schema in `infra/timescaledb/init.sql`:

```sql
CREATE EXTENSION IF NOT EXISTS timescaledb;
CREATE TABLE robot_state (
    ts        TIMESTAMPTZ NOT NULL,
    joints    DOUBLE PRECISION[] NOT NULL,   -- 6-element array
    ee_x      DOUBLE PRECISION NOT NULL,     -- flattened for cheap Grafana queries
    ee_y      DOUBLE PRECISION NOT NULL,
    ee_z      DOUBLE PRECISION NOT NULL
);
SELECT create_hypertable('robot_state', 'ts');
```

### 4.2 Latest-state aggregation → Redis

```python
set_latest_state(redis_client, state)   # SET state:latest <json(state)>
```

- One key (`state:latest`), overwritten every tick. Old values discarded.
- Serialization: JSON via `SimState.model_dump_json()`.
- Any client wanting "where is the arm *now*?" reads this one key — O(1), no time-range query needed.

Both writes are wrapped in a `try/except` that auto-reconnects on transient DB or Redis outages, so a container restart does not kill the telemetry loop.

## 5. Conversion to digital twin state — end-to-end proof

```
Omniverse arm moves (real sim, real physics)
  → sim-bridge publishes SimState @ 10 Hz         (stream)
  → telemetry SUBs and decodes into pydantic       (raw → typed)
  → 2 aggregations:
       ├── row per tick   → TimescaleDB           (historical state)
       └── overwrite key  → Redis                 (latest state)
  → Grafana queries TimescaleDB → visualizes state over time
  → Any client reads Redis → knows current state instantly
```

## 6. Live evidence

### 6.1 Grafana — real streams aggregated into the digital twin state

![Grafana healthy](./screenshots/grafana_dashboard_healthy.png)

*The **Pipeline health** panel shows `HEALTHY` — meaning ≥300 state messages received in the last minute (sim publishes at 10 Hz = 600/min).*

*The **Gripper position (cm)** panel shows the aggregated end-effector position derived from streamed joint state — Reach, Height, and Lateral traces over time.*

### 6.2 Grafana — pipeline health indicator flips when the stream stops

![Grafana DOWN](./screenshots/grafana_health_down.png)

*When sim-bridge stops publishing (Kit closed), the health indicator flips to `DOWN` within one minute — proving the aggregation is genuinely live, not cached.*

### 6.3 TimescaleDB — historical stream aggregated to rows

![Persistence before](./screenshots/persistence_before_timescale.png)

*46,570 rows aggregated from the sim stream (as of demo time), one row per publish tick.*

### 6.4 Redis — latest-state aggregation

![Persistence before Redis](./screenshots/persistence_before_redis.png)

*The `state:latest` key holds the most recent `SimState` as JSON — joints, ee_pose, timestamp.*

In [ ]:
# Read the live latest-state aggregate from Redis

import subprocess, json

result = subprocess.run(
    ["docker", "compose", "-f", "infra/docker-compose.yml",
     "exec", "-T", "redis", "redis-cli", "GET", "state:latest"],
    capture_output=True, text=True,
)
state = json.loads(result.stdout.strip())
print(json.dumps(state, indent=2))

# Expected output:
# {
#   "joints":  [0.0, 1.10, -0.55, 2.58, 0.0, 0.0],
#   "ee_pose": {"x": 39.99, "y": 19.75, "z": -0.0},
#   "ts":      1785503843.4560454
# }

## 7. Persistence proof — data + state both survive container restart

The rubric requires proving persistence in both **data** (historical) and **state** (latest). Same command run before and after `docker compose restart timescaledb redis`:

**Before restart:**

![Persistence before Timescale](./screenshots/persistence_before_timescale.png)

**After restart:**

![Persistence after Timescale](./screenshots/persistence_after_timescale.png)

Row count went from **46,570** to **47,100** — data survived AND new data kept arriving during the restart window (telemetry auto-reconnects). Latest state in Redis also survives:

![Persistence after Redis](./screenshots/persistence_after_redis.png)

*Byte-identical `state:latest` JSON before and after restart. Named Docker volumes on both `timescaledb-data` and `redis-data` guarantee durability.*

## 8. Tests proving the streams work

| Test | What it proves |
|------|---------------|
| `tests/integration/test_sim_to_telemetry.py` (Raziq-07) | Fake `SimState` via real Timescale + Redis containers lands a matching row AND updates Redis latest key |
| `tests/integration/test_actuation_mqtt.py` (Ibrohim-07) | Real MQTT round-trip against real mosquitto container — proves the outbound-to-physical-robot stream works |
| `tests/system/test_end_to_end.py` (A-07) | Full stack: `POST /command` → within 10 s → new row appears in `robot_state` with plausible joints |

## 9. Summary

- **Digital twin state** — formally defined as a pydantic v2 `SimState` model with 5 fields; contract-frozen v1.0.
- **Multiple real data streams** — 4 concurrent streams (user commands, LLM completions, ZMQ joint frames, ZMQ sim state), all real, all wired end-to-end.
- **Streaming acquisition** — HTTP (event-driven) + ZMQ PUSH/PULL (30 fps) + ZMQ PUB/SUB (10 Hz).
- **Aggregation** — telemetry dual-writes every incoming state message into TimescaleDB (history) and Redis (latest).
- **Persistence** — data (Timescale) + state (Redis) both survive `docker compose restart`, proven with before/after screenshots.
- **Observability** — Grafana `HEALTHY`/`DOWN` indicator proves the pipeline is live, not cached.